In [33]:
%load_ext autoreload
%autoreload 2

# Define autroreload so that it doesn't cause pain in the ass when we change the functions and run this notebook

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
import sys
from pathlib import Path
project_root = Path.cwd().resolve().parents[2]
sys.path.append(str(project_root))

from defs.diffusion.epsilon import *

In [35]:
# Set device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")  # Force CPU for testing purposes
print(f"Using device: {device}")

print("PyTorch version:", torch.__version__)
print("CUDA version PyTorch sees:", torch.version.cuda)
print("cuDNN enabled:", torch.backends.cudnn.enabled)
print("Number of GPUs detected:", torch.cuda.device_count())

Using device: cpu
PyTorch version: 2.5.1+cu121
CUDA version PyTorch sees: 12.1
cuDNN enabled: True
Number of GPUs detected: 0


In [36]:
# Define the dictionary of and initialize the MLP

cfg_model = {
    'time_embed': {
        'hidden_dim': 32,
        'hidden_n': 2
    },
    'ab_embed': {
        'hidden_dim': 48,
        'hidden_n': 3,
        'ab_dim': 3
    },
    'denoiser': {
        'hidden_dim': 128,
        'hidden_n': 4,
        'spec_dim': 81
    }
}

epsilon= Epsilon_MLP(cfg_model= cfg_model)
epsilon = epsilon.to(device)

In [37]:
# Testing the graph of the epsilon

# Create the test tensors
spec_tensor = torch.randn(5, 81).to(device=device, dtype=torch.float32)
time_tensor = torch.randn(5,1).to(device, dtype=torch.float32)
ab_tensor = torch.randn(5, 3).to(device, dtype=torch.float32)

output = epsilon(spec_tensor, time_tensor, ab_tensor)


# Create the graph:
from torchviz import make_dot
import graphviz
import os
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"

make_dot(output, params=dict(epsilon.named_parameters())).render("model_graph", format="png")

# This shows that the defined epsilon works as intended

'model_graph.png'

In [38]:
# We'll test the diffusion model here

from defs.diffusion.diffusion import *
from defs.diffusion.helper.noising.noising import *

scheduler = CosSchedule(1000)

ddpm = cond_diffusion(epsilon=epsilon, scheduler=scheduler)


In [39]:
x0_pred, noise, eps_pred = ddpm.training_procedure(spec_tensor, ab_tensor)

# cond_diffusion.training_procedure works without coding bugs

In [40]:
x_t, x_T = ddpm.sample(noise, ab_tensor)

donezo
